In [1]:
import geopandas as gpd
import pandas as pd

In [13]:
current_flu_year = 2026
cur_flu_yr_2dig = str(current_flu_year)[-2:]
flu_csv_path = "C:/Users/JKolberg/OneDrive - PSRC/GIS - Projects/FLU/Zoning_2026_d2_cleaned.csv"

old_flu_year = 2019
old_flu_yr_2dig = str(old_flu_year)[-2:]
old_flu_shp_path = "W:/gis/projects/compplan_zoning/flu19_reviewed.shp"

In [17]:
# get old xwalk for old FLU descriptions
old_xwalk_path = "J:/Staff/Christy/usim-baseyear/flu/Full_FLU_Master_Corres_File.xlsx"
old_xwalk = pd.read_excel(old_xwalk_path, sheet_name='Full Master FLU Corres File')[['FLUadj_Key','FLUadj_Definition']].dropna()

In [4]:
flu_table = pd.read_csv(flu_csv_path, encoding='latin-1')
flu_table = flu_table.drop(columns=[c for c in flu_table.columns if 'Unnamed' in c]).add_suffix(f'_{cur_flu_yr_2dig}')

In [30]:
old_flu = gpd.read_file(old_flu_shp_path).merge(old_xwalk, left_on=f'Juris_zn', right_on='FLUadj_Key', how='left')
old_flu_table = (
    old_flu
    .drop(columns=['geometry'])
    .copy()
    .add_suffix(f'_{old_flu_yr_2dig}')
)
old_flu_shp = old_flu[['Juris_zn','FLUadj_Definition','geometry']].copy()


c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 285 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(
c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 1768 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


In [6]:
old_flu_table.columns

Index(['Jurisdicti_19', 'Juris_zn_19', 'Zone_adj_19', 'Juris_zn_1_19',
       'Res_Use_19', 'Comm_Use_19', 'Office_Use_19', 'Indust_Use_19',
       'Mixed_Use_19', 'Public_Use_19', 'PUD_Use_19', 'MaxDU_Res_19',
       'MaxFAR_Com_19', 'MaxFAR_Off_19', 'MaxFAR_Ind_19', 'MaxDU_Mixe_19',
       'MaxFAR_Mix_19', 'MaxHt_Res_19', 'MaxHt_Mixe_19', 'MaxHt_Comm_19',
       'MaxHt_Offi_19', 'MaxHt_Indu_19', 'FLUadj_Key', 'FLUadj_Definition'],
      dtype='str')

In [7]:
flu_table.columns

Index(['Juris_26', 'Zone_26', 'juris_zn_26', 'Definition_26', 'Bonus_avail_26',
       'Bonus_included_26', 'Res_Use_26', 'Comm_Use_26', 'Office_Use_26',
       'Indust_Use_26', 'Mixed_Use_26', 'Public_Use_26', 'PUD_Use_26',
       'ResDU_lot_26', 'MinDU_Res_26', 'MinDU_Comm_26', 'MinDU_Office_26',
       'MinDU_Indust_26', 'MinDU_Mixed_26', 'MaxDU_Res_26', 'MaxDU_Comm_26',
       'MaxDU_Office_26', 'MaxDU_Indust_26', 'MaxDU_Mixed_26', 'MinFAR_Res_26',
       'MinFAR_Comm_26', 'MinFAR_Office_26', 'MinFAR_Indust_26',
       'MinFAR_Mixed_26', 'MaxFAR_Res_26', 'MaxFAR_Comm_26',
       'MaxFAR_Office_26', 'MaxFAR_Indust_26', 'MaxFAR_Mixed_26',
       'MaxHt_Res_26', 'MaxHt_Comm_26', 'MaxHt_Office_26', 'MaxHt_Indust_26',
       'MaxHt_Mixed_26', 'LC_Res_26', 'LC_Comm_26', 'LC_Office_26',
       'LC_Indust_26', 'LC_Mixed_26', 'rural_26', 'ADU notes_26'],
      dtype='str')

In [8]:
old_to_new_col_mapping = {
    'Jurisdicti_19': 'Juris_26',
    'Juris_zn_19': 'juris_zn_26',
    'FLUadj_Definition': 'Definition_26'
}

In [9]:
from difflib import SequenceMatcher
import re

def normalize_zone(s):
    """Normalize a zone string for near-matching: lowercase, strip, collapse separators."""
    s = str(s).strip().lower()
    s = re.sub(r'[-_\s,/]+', '', s)  # remove dashes, underscores, spaces, commas, slashes
    return s

# --- Build unique zone lists per jurisdiction for old (19) and new (26) ---

old_zones = (
    old_flu_table[['Jurisdicti_19', 'Juris_zn_19', 'FLUadj_Definition']]
    .drop_duplicates()
    .rename(columns={'Jurisdicti_19': 'jurisdiction', 'Juris_zn_19': 'zone_19', 'FLUadj_Definition': 'desc_19'})
)

new_zones = (
    flu_table[['Juris_26', 'juris_zn_26', 'Definition_26']]
    .drop_duplicates()
    .rename(columns={'Juris_26': 'jurisdiction', 'juris_zn_26': 'zone_26', 'Definition_26': 'desc_26'})
    .drop_duplicates(subset=['jurisdiction', 'zone_26'], keep='first')
)

print(f"Old zones: {len(old_zones)} unique rows")
print(f"New zones: {len(new_zones)} unique rows")

Old zones: 1919 unique rows
New zones: 1644 unique rows


In [10]:
crosswalk_rows = []

# Jurisdiction alias mapping: normalized form -> canonical normalized form
# Note: "snohomish" (city) is intentionally NOT mapped to "snohomishcounty"
jurisdiction_aliases = {
    'mlt': 'mountlaketerrace',
    'bainbridge': 'bainbridgeisland',
    'mercer': 'mercerisland',
    'up': 'universityplace',
    'snoco': 'snohomishcounty',
    'pierce': 'piercecounty',
    'kitsap': 'kitsapcounty',
    'king': 'kingcounty',
}

def normalize_jurisdiction(s):
    """Normalize jurisdiction name, applying known aliases."""
    norm = normalize_zone(s)
    return jurisdiction_aliases.get(norm, norm)

# Normalize jurisdiction names for grouping (handles "Black Diamond" vs "BlackDiamond" etc.)
new_zones['jurisdiction_norm'] = new_zones['jurisdiction'].apply(normalize_jurisdiction)
old_zones['jurisdiction_norm'] = old_zones['jurisdiction'].apply(normalize_jurisdiction)

# Iterate over new (26) jurisdictions to ensure every new zone is in the crosswalk
for juris_norm in new_zones['jurisdiction_norm'].unique():
    new_j = new_zones[new_zones['jurisdiction_norm'] == juris_norm].copy()
    old_j = old_zones[old_zones['jurisdiction_norm'] == juris_norm].copy()
    juris = new_j['jurisdiction'].iloc[0]  # use the new jurisdiction name as canonical
    
    matched_old = set()
    matched_new = set()
    
    # --- Pass 1: Exact match on Juris_zn ---
    for ni, nrow in new_j.iterrows():
        for oi, orow in old_j.iterrows():
            if oi in matched_old:
                continue
            if str(nrow['zone_26']).strip() == str(orow['zone_19']).strip():
                crosswalk_rows.append({
                    'jurisdiction': juris,
                    'zone_19': orow['zone_19'], 'desc_19': orow['desc_19'],
                    'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                    'match_type': 'exact', 'confidence': 1.0
                })
                matched_old.add(oi)
                matched_new.add(ni)
                break

    # --- Pass 2: Normalized near-match on Juris_zn (dashes/underscores/case) ---
    remaining_new = new_j[~new_j.index.isin(matched_new)]
    remaining_old = old_j[~old_j.index.isin(matched_old)]
    
    for ni, nrow in remaining_new.iterrows():
        norm_new = normalize_zone(nrow['zone_26'])
        for oi, orow in remaining_old.iterrows():
            if oi in matched_old:
                continue
            if norm_new == normalize_zone(orow['zone_19']):
                crosswalk_rows.append({
                    'jurisdiction': juris,
                    'zone_19': orow['zone_19'], 'desc_19': orow['desc_19'],
                    'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                    'match_type': 'near_match', 'confidence': 0.8
                })
                matched_old.add(oi)
                matched_new.add(ni)
                break

    # --- Pass 3: Fuzzy match on zone name (for remaining new zones) ---
    remaining_new = new_j[~new_j.index.isin(matched_new)]
    remaining_old = old_j[~old_j.index.isin(matched_old)]
    
    for ni, nrow in remaining_new.iterrows():
        best_score = 0.0
        best_oi = None
        best_orow = None
        
        for oi, orow in remaining_old.iterrows():
            if oi in matched_old:
                continue
            
            zone_sim = SequenceMatcher(None, normalize_zone(nrow['zone_26']), normalize_zone(orow['zone_19'])).ratio()
            
            if zone_sim > best_score:
                best_score = zone_sim
                best_oi = oi
                best_orow = orow
        
        if best_score >= 0.4 and best_orow is not None:
            crosswalk_rows.append({
                'jurisdiction': juris,
                'zone_19': best_orow['zone_19'], 'desc_19': best_orow['desc_19'],
                'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                'match_type': 'fuzzy', 'confidence': round(best_score, 3)
            })
            matched_old.add(best_oi)
        else:
            # New zone with no old match — still included for completeness
            crosswalk_rows.append({
                'jurisdiction': juris,
                'zone_19': None, 'desc_19': None,
                'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                'match_type': 'new_zone', 'confidence': 0.0
            })

crosswalk = pd.DataFrame(crosswalk_rows)
print(crosswalk['match_type'].value_counts())
print(f"\nTotal crosswalk rows: {len(crosswalk)}")
print(f"Total new zones: {len(new_zones)} — all accounted for: {len(crosswalk) == len(new_zones)}")

match_type
exact         562
fuzzy         540
near_match    398
new_zone      144
Name: count, dtype: int64

Total crosswalk rows: 1644
Total new zones: 1644 — all accounted for: True


In [11]:
# Flag rows needing manual review: fuzzy matches and new zones with no old equivalent
crosswalk['needs_review'] = crosswalk['match_type'].isin(['fuzzy', 'new_zone'])

# Add a column for manual override
crosswalk['manual_match'] = ''

# Add a column to confirm a zone is genuinely new (clears needs_review)
crosswalk['confirmed_new'] = False

# Show summary
print("=== Match Summary ===")
print(crosswalk.groupby('match_type')['confidence'].describe()[['count','mean','min','max']])
print(f"\nRows needing manual review: {crosswalk['needs_review'].sum()}")
print(f"Rows auto-matched (exact + near): {(~crosswalk['needs_review']).sum()}")

=== Match Summary ===
            count      mean  min   max
match_type                            
exact       562.0  1.000000  1.0  1.00
fuzzy       540.0  0.731687  0.4  0.96
near_match  398.0  0.800000  0.8  0.80
new_zone    144.0  0.000000  0.0  0.00

Rows needing manual review: 684
Rows auto-matched (exact + near): 960


In [12]:
# View only the rows that need manual review, sorted by jurisdiction then confidence
review_df = (
    crosswalk[crosswalk['needs_review']]
    .sort_values(['jurisdiction', 'match_type', 'confidence'])
    [['jurisdiction', 'zone_19', 'desc_19', 'zone_26', 'desc_26', 'match_type', 'confidence', 'manual_match']]
)
print(f"Rows for manual review: {len(review_df)}")
review_df

Rows for manual review: 684


,jurisdiction,zone_19,desc_19,zone_26,desc_26,match_type,confidence,manual_match
1,Algona,Algona_C2vacantstreams,NaN,Algona_C-2,The C-2 general commercial district is intende...,fuzzy,0.552,
2,Algona,Algona_C3vacantstreams,NaN,Algona_C-3,The C-3 heavy commercial district is intended ...,fuzzy,0.552,
3,Algona,Algona_M1vacantstreams,NaN,Algona_M-1,Light industrial zones are intended for light ...,fuzzy,0.552,
4,Algona,Algona_HEAVYCOMMERCIAL,NaN,Algona_R-L,The R-L low density residential district is in...,fuzzy,0.552,
5,Algona,Algona_C1vacantstreams,NaN,Algona_R-M,The R-M medium density residential district is...,fuzzy,0.552,
...,...,...,...,...,...,...,...,...
1640,Woodway,Woodway_FRP R-87,NaN,Woodway_R-87,The primary function of the residential R-87 z...,fuzzy,0.870,
1638,Woodway,Woodway_SR R-14.5,NaN,Woodway_R-14.5,The primary function of the residential R-14.5...,fuzzy,0.923,
1641,Woodway,NaN,NaN,Woodway_UV,The purpose of the urban village (UV) zone is ...,new_zone,0.000,
1643,Yarrow_Point,YarrowPoint_Right of Way,NaN,Yarrow_Point_R-15,The purpose of this title is to regulate the u...,fuzzy,0.686,


In [15]:
# Read manual adjustments from Excel and apply to crosswalk
manual = pd.read_excel('flu_crosswalk_19_to_26.xlsx')
now = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
manual.to_excel(f'flu_crosswalk_19_to_26_backup_{now}.xlsx', index=False)  # backup before applying edits

# Apply confirmed_new flags from the Excel
if 'confirmed_new' in manual.columns:
    confirmed = manual[manual['confirmed_new'].astype(str).str.strip().str.upper().isin(['TRUE', '1', 'YES'])]
    for _, row in confirmed.iterrows():
        mask = (crosswalk['jurisdiction'] == row['jurisdiction']) & (crosswalk['zone_26'] == row['zone_26'])
        if mask.any():
            crosswalk.loc[mask, 'confirmed_new'] = True
            crosswalk.loc[mask, 'needs_review'] = False
            print(f"  Confirmed new: {row['jurisdiction']} / {row['zone_26']}")
    print(f"Applied {len(confirmed)} confirmed_new flags")

# Apply manual overrides where manual_match is non-empty
manual_edits = manual[manual['manual_match'].notna() & (manual['manual_match'].astype(str).str.strip() != '')]

print(f"Found {len(manual_edits)} manual adjustments")

for _, row in manual_edits.iterrows():
    mask = (crosswalk['jurisdiction'] == row['jurisdiction']) & (crosswalk['zone_26'] == row['zone_26'])
    if mask.any():
        manual_val = str(row['manual_match']).strip()
        crosswalk.loc[mask, 'manual_match'] = manual_val
        # Look up the old zone info from old_zones
        old_match = old_zones[old_zones['zone_19'] == manual_val]
        if not old_match.empty:
            crosswalk.loc[mask, 'zone_19'] = manual_val
            crosswalk.loc[mask, 'desc_19'] = old_match.iloc[0]['desc_19']
            crosswalk.loc[mask, 'match_type'] = 'manual'
            crosswalk.loc[mask, 'confidence'] = 1.0
            crosswalk.loc[mask, 'needs_review'] = False
        print(f"  Updated: {row['jurisdiction']} / {row['zone_26']} -> {manual_val}")
    else:
        print(f"  WARNING: No match in crosswalk for {row['jurisdiction']} / {row['zone_26']}")

print(f"\n=== Updated Match Summary ===")
print(crosswalk['match_type'].value_counts())
print(f"Rows still needing review: {crosswalk['needs_review'].sum()}")

Applied 0 confirmed_new flags
Found 315 manual adjustments
  Updated: Algona / Algona_C-1 -> Algona_MIXEDUSECOMMERCIAL
  Updated: Algona / Algona_C-2 -> Algona_GENERALCOMMERCIAL
  Updated: Algona / Algona_C-3 -> Algona_HEAVYCOMMERCIAL
  Updated: Algona / Algona_M-1 -> Algona_LIGHTINDUSTRIAL
  Updated: Algona / Algona_R-L -> Algona_LOWDENSITYRESIDENTIAL
  Updated: Algona / Algona_R-M -> Algona_MEDIUMDENSITYRESIDENTIAL
  Updated: Arlington / Arlington_PSP -> Arlington_P/SP
  Updated: Arlington / Arlington_AP-ISZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-ITZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-OSZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-RPZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-SSZ -> Arlington_AF
  Updated: Arlington / Arlington_GC_MXD -> Arlington_GC
  Updated: Arlington / Arlington_HC_MXD -> Arlington_HC
  Updated: Arlington / Arlington_RHC -> Arlington_RHD
  Updated: Arlington / Arlington_RMC -> Arlington_RMD
  Updated: Auburn / Aubu

In [14]:
# Export: full crosswalk, review-only subset, and old zones to Excel
crosswalk.to_excel('flu_crosswalk_19_to_26.xlsx', index=False)
old_zones.to_excel('old_zones_19.xlsx', index=False)
print("Saved flu_crosswalk_19_to_26.xlsx, and old_zones_19.xlsx")

Saved flu_crosswalk_19_to_26.xlsx, and old_zones_19.xlsx


In [29]:
flu_shp_gdb = "C:/Users/JKolberg/OneDrive - PSRC/GIS - Projects/FLU/FLU_draft2.gdb"
flu_shp_layer = 'FLU2025_cleaned'
flu_shp = gpd.read_file(flu_shp_gdb, layer=flu_shp_layer)[['Juris_zn','Definition','geometry']]

In [31]:
# Spatial crosswalk: match new zones to old zones by area overlap

# Ensure same CRS
old_aligned = old_flu_shp.to_crs(flu_shp.crs) if flu_shp.crs != old_flu_shp.crs else old_flu_shp

# Dissolve to one (multi)polygon per unique zone
new_zones_geo = flu_shp.dissolve(by='Juris_zn').reset_index()[['Juris_zn', 'geometry']]
old_zones_geo = old_aligned.dissolve(by='Juris_zn').reset_index()[['Juris_zn', 'geometry']]

new_zones_geo = new_zones_geo.rename(columns={'Juris_zn': 'Juris_zn_26'})
old_zones_geo = old_zones_geo.rename(columns={'Juris_zn': 'Juris_zn_19'})

# Total area of each new zone (for computing percentages)
new_zones_geo['new_area'] = new_zones_geo.geometry.area

# Intersection overlay
overlay = gpd.overlay(new_zones_geo, old_zones_geo, how='intersection', keep_geom_type=True)
overlay['intersection_area'] = overlay.geometry.area
overlay['pct_overlap'] = overlay['intersection_area'] / overlay['new_area']

# Best match per new zone: old zone with the largest overlap share
spatial_xwalk = (
    overlay
    .sort_values('pct_overlap', ascending=False)
    .drop_duplicates(subset='Juris_zn_26', keep='first')
    [['Juris_zn_26', 'Juris_zn_19', 'pct_overlap']]
    .sort_values('pct_overlap')
    .reset_index(drop=True)
)

OVERLAP_CUTOFF = 0.5
spatial_xwalk['likely_match'] = spatial_xwalk['pct_overlap'] >= OVERLAP_CUTOFF

print(f"Total new zones: {len(spatial_xwalk)}")
print(f"Matches above {OVERLAP_CUTOFF:.0%} cutoff: {spatial_xwalk['likely_match'].sum()}")
print(f"Below cutoff (needs review): {(~spatial_xwalk['likely_match']).sum()}")
spatial_xwalk

Total new zones: 1549
Matches above 50% cutoff: 1455
Below cutoff (needs review): 94


,Juris_zn_26,Juris_zn_19,pct_overlap,likely_match
0,Stanwood_NZ,Snoco_R-5,0.000005,False
1,Issaquah_UVSF-0,Issaquah_UVSF-1,0.007371,False
2,Woodinville_HWY,Woodinville_GB,0.009671,False
3,King_County_,King_RA-10,0.035749,False
4,Gig_Harbor_No_Zoning,GigHarbor_C-1,0.035762,False
...,...,...,...,...
1544,Kitsap_MRONC,Kitsap_IND,1.000000,True
1545,Black_Diamond_R12-TDR,BlackDiamond_MPD,1.000000,True
1546,Everett_MU25,Everett_UM,1.000000,True
1547,Kitsap_MROUL,Kitsap_IND,1.000000,True
